The notebook tests blender rendering pipeline, then tests the plibs save and load (png and qoi)

In [ ]:
import json

import torch

%load_ext autoreload
%autoreload 2

# we assume plibs is outside blender_rendering/ and in the same root_dir
import sys

repo_root = ".."
sys.path.insert(0, repo_root)  # repo root
sys.path.insert(0, "../..")  # for plibs at same level as repo root

import os
import shutil
from timeit import default_timer as timer

import blender_open3d_utils
import blender_plib_utils
import numpy as np
import open3d as o3d
import utils

from plibs import exr_utils, json_utils, render as p_render, rigid_motion, structures, utils as p_utils

In [ ]:
# load mesh

test_mode = "dynamic_two_objs"  # "dynamic"  # static

if test_mode == "static":
    mesh_filename = os.path.join(repo_root, "assets/0a0f1b107acb4b94a8211e11ab69a67f.glb")
    num_frames = 1
    frame_skip = 1
    dynamic = False
elif test_mode == "dynamic":
    mesh_filename = os.path.join(repo_root, "assets/dandys_world_goob_rig__animation.glb")
    num_frames = 4
    frame_skip = 6
    dynamic = True
elif test_mode == "static_two_objs":
    mesh_filename = [
        os.path.join(repo_root, "assets/0a0f1b107acb4b94a8211e11ab69a67f.glb"),
        os.path.join(repo_root, "assets/boom_karts_rocket_vehicle.glb"),
    ]
    num_frames = 1
    frame_skip = 1
    dynamic = False
elif test_mode == "dynamic_two_objs":
    mesh_filename = [
        os.path.join(repo_root, "assets/boom_karts_rocket_vehicle.glb"),
        os.path.join(repo_root, "assets/dandys_world_goob_rig__animation.glb"),
    ]
    num_frames = 4
    frame_skip = 6
    dynamic = True
else:
    raise ValueError(f"Invalid mode: {test_mode}")


# generate random cameras
H_c2w_o3d = rigid_motion.generate_random_camera_poses_lookat(
    n=10,
    pinhole_min_r=3,
    pinhole_max_r=5,
    lookat_r=0.1,
    up_method="y",
    invert_y=True,
)  # (n, 4, 4)
H_c2w_o3d = H_c2w_o3d.cpu().numpy()


w, h = 512, 1024
fov = np.random.rand(H_c2w_o3d.shape[0]) * 10 + 30  # (n,)
intrinsic_o3d = torch.from_numpy(
    p_render.derive_camera_intrinsics(
        width_px=w,
        height_px=h,
        fov=fov,
    )
)  # (n, 3, 3)

# add a random cx cy
rcx = w / 2 * (np.random.rand(H_c2w_o3d.shape[0]) - 0.5) * 2 * 0.2  # (n,)
rcy = h / 2 * (np.random.rand(H_c2w_o3d.shape[0]) - 0.5) * 2 * 0.2  # (n,)

intrinsic_o3d[:, 0, 2] += torch.from_numpy(rcx)
intrinsic_o3d[:, 1, 2] += torch.from_numpy(rcy)

In [ ]:
# construct the config dict
mesh_dicts = []
if isinstance(mesh_filename, str):
    mdict = dict(
        name="mesh",
        filename=mesh_filename,
        normalize_first=True,
        H_c2w=np.eye(4),
        scale=np.array([1.0, 1.0, 1.0]),
    )
    mesh_dicts.append(mdict)
else:
    # randomly place the meshes in the scene
    # blender is z-up and so do most glbs
    for filename in mesh_filename:
        _H_c2w = (
            rigid_motion.generate_random_camera_poses_lookat(
                n=1,
                pinhole_min_r=0,
                pinhole_max_r=1,
                lookat_r=0.25,
                up_method="z",
                invert_y=False,
            )
            .squeeze(0)
            .detach()
            .cpu()
            .numpy()
        )  # (4, 4)
        _scale = np.random.rand(1) * 1.5 + 0.5  # (1,)
        mdict = dict(
            name="mesh",
            filename=filename,
            normalize_first=True,
            H_c2w=_H_c2w,
            scale=[_scale.item() for _ in range(3)],
        )
        mesh_dicts.append(mdict)


camera_dicts = []
for ii in range(H_c2w_o3d.shape[0]):
    mdict = blender_open3d_utils.convert_open3d_camera_to_blender(
        H_c2w=H_c2w_o3d[ii],
        intrinsic=intrinsic_o3d[ii],
        width_px=w,
        height_px=h,
    )
    camera_dicts.append(mdict)


light_dicts = []
mdict = dict(
    name="light 1",
    light_type="SUN",
    H_c2w=[
        [1.0, 0.0, 0.0, 0],
        [0.0, 1.0, 0.0, 0],
        [0.0, 0.0, 1.0, 5.0],
        [0.0, 0.0, 0.0, 1.0],
    ],
    energy=5,
    use_shadow=False,
    specular_factor=1.0,
)
light_dicts.append(mdict)

config_dict = dict(
    meshes=mesh_dicts,
    cameras=camera_dicts,
    lighting=light_dicts,
)

json_filename = "tmp.json"
with open(json_filename, "w") as f:
    json.dump(config_dict, f, indent=2, cls=json_utils.NumpyJsonEncoder)

In [ ]:
# render with blender
out_dir = os.path.join("tmp_output")

if os.path.exists(out_dir):
    shutil.rmtree(out_dir)

blender_script_version = "v3"
if blender_script_version == "v1":
    blender_script_name = os.path.join(repo_root, "blender_utils.py")
elif blender_script_version == "v2":
    blender_script_name = os.path.join(repo_root, "blender_utils_v2.py")
elif blender_script_version == "v3":
    blender_script_name = os.path.join(repo_root, "blender_utils_v3.py")
else:
    raise NotImplementedError(f"blender_script_version = {blender_script_version} not implemented")


blender_cmd = utils.get_blender_exe()
cmd = (
    f"{blender_cmd} --background --python {blender_script_name} -- "
    f"--filename {json_filename} --out_dir {out_dir} "
    f"--num_frames {num_frames} "
    f"--frame_skip {frame_skip} "
    f"--dynamic {int(dynamic)} "
    f"--normalize_entire_scene 1 "
    f"--device GPU"
)

print(cmd)
os.system(cmd)
print(f"output is written to {out_dir}")

In [ ]:
# read blender rendered results
# use plibs
rgbd = blender_plib_utils.read_blender_results_to_rgbd(
    result_dir=out_dir,
    from_idx=0,
    to_idx=None,
    from_bidx=0,
    to_bidx=None,
    use_srgb=True,
    flag_save_space=False,
    dynamic=None,  # auto-detect
    th_alpha=0.5,
    min_depth=0.0,
    max_depth=1.0e4,
    fps=24,
)
print(f"rgbd.shape: {rgbd.shape}")

In [ ]:
# plot point cloud
world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1)
cam_frames = rgbd.camera.get_camera_frames()[0]
point_cloud = rgbd.get_pcd()

o3d_pcds = point_cloud.get_o3d_pcds(use_normal_for_color=True)
p_utils.visualize_mesh_sequence(meshes=o3d_pcds, static_meshes=[world_frame] + cam_frames)

In [ ]:
# save rgbd
for mode in ["png", "qoi"]:
    print(f"checking {mode}..", flush=True)
    rgbd_out_dir = os.path.join(out_dir, f"rgbd_{mode}")
    if os.path.exists(rgbd_out_dir):
        shutil.rmtree(rgbd_out_dir)

    print(f"  saving..", flush=True)
    stime = timer()
    rgbd.save_as(
        out_dir=rgbd_out_dir,
        overwrite=True,
        mode=mode,  # 'npy', 'exr', 'png', 'qoi'
        background_color=0,
        save_attr_names=["rgb", "depth", "normal_w", "hit_map"],
        flag_save_space=True,
    )
    print(f"    used {timer() - stime:.2f} seconds.", flush=True)

    # load
    print(f"  loading..", flush=True)
    stime = timer()
    rgbd_tmp = rgbd.load_from(
        index_filename=os.path.join(rgbd_out_dir, "index.json"),
    )
    print(f"    used {timer() - stime:.2f} seconds.", flush=True)

    # check hit_map, alpha
    assert torch.allclose(rgbd.hit_map, rgbd_tmp.hit_map)
    assert torch.allclose(rgbd.other_maps["alpha"], rgbd_tmp.other_maps["alpha"], rtol=1e-3, atol=1e-3)

    # depth
    for attr_name in ["depth"]:
        arr1 = getattr(rgbd, attr_name, None)
        arr2 = getattr(rgbd_tmp, attr_name, None)
        if arr1 is not None and arr2 is not None:
            arr1 = arr1.float() * rgbd.hit_map.float()
            arr2 = arr2.float() * rgbd_tmp.hit_map.float()
            assert torch.allclose(
                arr1,
                arr2,
                rtol=1e-3,
                atol=1e-3,
            ), (
                f"{attr_name}: "
                f"max: {arr1.max()} {arr2.max()}, "
                f"min: {arr1.min()} {arr2.min()}, "
                f"max_diff: {(arr1 - arr2).abs().max()}"
            )

    # rgb, normal
    for attr_name in ["rgb", "normal_w"]:
        arr1 = getattr(rgbd, attr_name, None)
        arr2 = getattr(rgbd_tmp, attr_name, None)
        if arr1 is not None and arr2 is not None:
            arr1 = arr1.float() * rgbd.hit_map.unsqueeze(-1).float()
            arr2 = arr2.float() * rgbd_tmp.hit_map.unsqueeze(-1).float()
            if attr_name == "normal_w":
                assert ((torch.linalg.vector_norm(arr2[rgbd_tmp.hit_map], dim=-1) - 1).abs() < 1e-6).all()
                rtol = 1e-2  # 8bit
                atol = 1e-2
                diff_angle = (
                    torch.arccos((arr1[rgbd.hit_map] * arr2[rgbd_tmp.hit_map]).sum(dim=-1).clamp(max=1, min=-1))
                    * 180
                    / torch.pi
                )
                print(
                    f"normal diff angle: "
                    f"max: {diff_angle.abs().max():.2f} degrees, "
                    f"mean: {diff_angle.abs().mean():.2f} degrees, "
                    f"std: {diff_angle.abs().std():.2f} degrees"
                )
            else:
                rtol = 1e-5
                atol = 1e-5
            assert torch.allclose(
                arr1,
                arr2,
                rtol=rtol,
                atol=atol,
            ), (
                f"{attr_name}: "
                f"max: {arr1.max()} {arr2.max()}, "
                f"min: {arr1.min()} {arr2.min()}, "
                f"max_diff: {(arr1 - arr2).abs().max()}"
            )

In [ ]:
# save rgbd flat
for mode in ["png", "qoi"]:
    print(f"checking {mode}..", flush=True)
    rgbd_out_dir = os.path.join(out_dir, f"rgbd_flat_{mode}")
    if os.path.exists(rgbd_out_dir):
        shutil.rmtree(rgbd_out_dir)

    print(f"  saving..", flush=True)
    stime = timer()
    rgbd.save_as_flat(
        out_dir=rgbd_out_dir,
        prefix="obj",
        overwrite=True,
        background_color=0,
        mode=mode,
        save_attr_names=["rgb", "depth", "normal_w", "hit_map"],
    )
    print(f"    used {timer() - stime:.2f} seconds.", flush=True)

    # load
    print(f"  loading..", flush=True)
    stime = timer()
    rgbd_tmp = rgbd.load_from_flat(
        index_filename=os.path.join(rgbd_out_dir, "obj.index.json"),
    )
    print(f"    used {timer() - stime:.2f} seconds.", flush=True)

    # check hit_map, alpha
    assert torch.allclose(rgbd.hit_map, rgbd_tmp.hit_map)
    assert torch.allclose(rgbd.other_maps["alpha"], rgbd_tmp.other_maps["alpha"], rtol=1e-3, atol=1e-3)

    # depth
    for attr_name in ["depth"]:
        arr1 = getattr(rgbd, attr_name, None)
        arr2 = getattr(rgbd_tmp, attr_name, None)
        if arr1 is not None and arr2 is not None:
            arr1 = arr1.float() * rgbd.hit_map.float()
            arr2 = arr2.float() * rgbd_tmp.hit_map.float()
            assert torch.allclose(
                arr1,
                arr2,
                rtol=1e-3,
                atol=1e-3,
            ), (
                f"{attr_name}: "
                f"max: {arr1.max()} {arr2.max()}, "
                f"min: {arr1.min()} {arr2.min()}, "
                f"max_diff: {(arr1 - arr2).abs().max()}"
            )

    # rgb, normal
    for attr_name in ["rgb", "normal_w"]:
        arr1 = getattr(rgbd, attr_name, None)
        arr2 = getattr(rgbd_tmp, attr_name, None)
        if arr1 is not None and arr2 is not None:
            arr1 = arr1.float() * rgbd.hit_map.unsqueeze(-1).float()
            arr2 = arr2.float() * rgbd_tmp.hit_map.unsqueeze(-1).float()
            if attr_name == "normal_w":
                assert ((torch.linalg.vector_norm(arr2[rgbd_tmp.hit_map], dim=-1) - 1).abs() < 1e-6).all()
                rtol = 1e-2  # 8bit
                atol = 1e-2
                diff_angle = (
                    torch.arccos((arr1[rgbd.hit_map] * arr2[rgbd_tmp.hit_map]).sum(dim=-1).clamp(max=1, min=-1))
                    * 180
                    / torch.pi
                )
                print(
                    f"normal diff angle: "
                    f"max: {diff_angle.abs().max():.2f} degrees, "
                    f"mean: {diff_angle.abs().mean():.2f} degrees, "
                    f"std: {diff_angle.abs().std():.2f} degrees"
                )
            else:
                rtol = 1e-5
                atol = 1e-5
            assert torch.allclose(
                arr1,
                arr2,
                rtol=rtol,
                atol=atol,
            ), (
                f"{attr_name}: "
                f"max: {arr1.max()} {arr2.max()}, "
                f"min: {arr1.min()} {arr2.min()}, "
                f"max_diff: {(arr1 - arr2).abs().max()}"
            )

In [ ]:
attr_name = "normal_w"
arr1 = getattr(rgbd, attr_name, None)
arr2 = getattr(rgbd_tmp, attr_name, None)
if arr1 is not None and arr2 is not None:
    arr1 = arr1.float() * rgbd.hit_map.unsqueeze(-1).float()
    arr2 = arr2.float() * rgbd_tmp.hit_map.unsqueeze(-1).float()
torch.arccos((arr1[rgbd.hit_map] * arr2[rgbd_tmp.hit_map]).sum(dim=-1).clamp(max=1, min=-1)) * 180 / torch.pi